In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# used to remove the old processed dataset if performing the split again
import shutil

shutil.rmtree(
    "/content/drive/MyDrive/SIH_Crop_Project/dataset/processed_dataset",
    ignore_errors=True
)

In [ ]:
import os
import random
import shutil
from pathlib import Path

# =========================
# CONFIG
# =========================

SOURCE_DIR = "/content/drive/MyDrive/SIH_Crop_Project/dataset/Plant_Village_custom"

OUTPUT_DIR = "/content/drive/MyDrive/SIH_Crop_Project/dataset/processed_dataset"

TRAIN_SPLIT = 0.70
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

SEED = 42

# =========================
# REPRODUCIBILITY
# =========================

random.seed(SEED)

# =========================
# CREATE OUTPUT FOLDERS
# =========================

for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(OUTPUT_DIR, split), exist_ok=True)

# =========================
# PROCESS EACH CLASS
# =========================

classes = os.listdir(SOURCE_DIR)

for class_name in classes:

    class_path = os.path.join(SOURCE_DIR, class_name)

    if not os.path.isdir(class_path):
        continue

    images = os.listdir(class_path)

    # Shuffle images randomly
    random.shuffle(images)

    total_images = len(images)

    # Split indices
    train_end = int(total_images * TRAIN_SPLIT)
    val_end = train_end + int(total_images * VAL_SPLIT)

    train_images = images[:train_end]
    val_images = images[train_end:val_end]
    test_images = images[val_end:]

    split_map = {
        "train": train_images,
        "val": val_images,
        "test": test_images
    }

    # =========================
    # COPY FILES
    # =========================

    for split_name, split_images in split_map.items():

        split_class_dir = os.path.join(
            OUTPUT_DIR,
            split_name,
            class_name
        )

        os.makedirs(split_class_dir, exist_ok=True)

        for image_name in split_images:

            src = os.path.join(class_path, image_name)

            dst = os.path.join(split_class_dir, image_name)

            shutil.copy2(src, dst)

    # =========================
    # PRINT STATS
    # =========================

    print(f"\nClass: {class_name}")
    print(f"Total: {total_images}")
    print(f"Train: {len(train_images)}")
    print(f"Val: {len(val_images)}")
    print(f"Test: {len(test_images)}")

print("\nDataset split completed successfully.")


Class: Tomato___Leaf_Mold
Total: 952
Train: 666
Val: 142
Test: 144

Class: Potato___Early_blight
Total: 1000
Train: 700
Val: 150
Test: 150

Class: Potato___healthy
Total: 152
Train: 106
Val: 22
Test: 24

Class: Tomato___healthy
Total: 1592
Train: 1114
Val: 238
Test: 240

Class: Tomato___Early_blight
Total: 1000
Train: 700
Val: 150
Test: 150

Class: Potato___Late_blight
Total: 1000
Train: 700
Val: 150
Test: 150

Class: Apple___Apple_scab
Total: 630
Train: 441
Val: 94
Test: 95

Class: Apple___Black_rot
Total: 621
Train: 434
Val: 93
Test: 94

Class: Apple___Cedar_apple_rust
Total: 275
Train: 192
Val: 41
Test: 42

Class: Apple___healthy
Total: 1645
Train: 1151
Val: 246
Test: 248

Class: Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
Total: 513
Train: 359
Val: 76
Test: 78

Class: Tomato___Septoria_leaf_spot
Total: 1771
Train: 1239
Val: 265
Test: 267

Class: Corn_(maize)___Northern_Leaf_Blight
Total: 985
Train: 689
Val: 147
Test: 149

Class: Corn_(maize)___Common_rust_
Total: 1192
Train

In [ ]:
import os

DATASET_PATH = "/content/drive/MyDrive/SIH_Crop_Project/dataset/processed_dataset"

splits = ["train", "val", "test"]

grand_total = 0

for split in splits:

    split_path = os.path.join(DATASET_PATH, split)

    total_images = 0

    print(f"\n===== {split.upper()} =====")

    classes = os.listdir(split_path)

    for class_name in classes:

        class_path = os.path.join(split_path, class_name)

        num_images = len(os.listdir(class_path))

        total_images += num_images

        print(f"{class_name}: {num_images}")

    grand_total += total_images

    print(f"\nTotal {split} images: {total_images}")

print(f"\nTOTAL DATASET IMAGES: {grand_total}")


===== TRAIN =====
Tomato___Leaf_Mold: 666
Potato___Early_blight: 700
Potato___healthy: 106
Tomato___healthy: 1114
Tomato___Early_blight: 700
Potato___Late_blight: 700
Apple___Apple_scab: 441
Apple___Black_rot: 434
Apple___Cedar_apple_rust: 192
Apple___healthy: 1151
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot: 359
Tomato___Septoria_leaf_spot: 1239
Corn_(maize)___Northern_Leaf_Blight: 689
Corn_(maize)___Common_rust_: 834
Corn_(maize)___healthy: 813
Grape___Black_rot: 826
Grape___healthy: 296
Grape___Leaf_blight_(Isariopsis_Leaf_Spot): 753
Grape___Esca_(Black_Measles): 968

Total train images: 12981

===== VAL =====
Tomato___Leaf_Mold: 142
Potato___Early_blight: 150
Potato___healthy: 22
Tomato___healthy: 238
Tomato___Early_blight: 150
Potato___Late_blight: 150
Apple___Apple_scab: 94
Apple___Black_rot: 93
Apple___Cedar_apple_rust: 41
Apple___healthy: 246
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot: 76
Tomato___Septoria_leaf_spot: 265
Corn_(maize)___Northern_Leaf_Blight: 147


In [ ]:
#corrupted image detection
from PIL import Image
import os

DATASET_PATH = "/content/drive/MyDrive/SIH_Crop_Project/dataset/processed_dataset"

corrupted_images = []

splits = ["train", "val", "test"]

for split in splits:

    split_path = os.path.join(DATASET_PATH, split)

    classes = os.listdir(split_path)

    print(f"\nChecking {split} set...")

    for class_name in classes:

        class_path = os.path.join(split_path, class_name)

        images = os.listdir(class_path)

        for image_name in images:

            image_path = os.path.join(class_path, image_name)

            try:
                with Image.open(image_path) as img:
                    img.verify()

            except Exception:

                corrupted_images.append(image_path)

                print(f"Corrupted: {image_path}")

print("\n===== FINAL REPORT =====")

print(f"Total corrupted images: {len(corrupted_images)}")


Checking train set...

Checking val set...

Checking test set...

===== FINAL REPORT =====
Total corrupted images: 0
